<a href="https://colab.research.google.com/github/JohnWendelG/JohnWendelG/blob/main/_Programa_integrador_de_automatizaci%C3%B3n_vf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

REFACTORIZA CODIGOS Y SUBPROCESOS MESCLADOS EN PROCESOS DE INVENTARIO FINAL E INVENTARIO DE TRIBUS

Se separan los codigos de las columnas correspondientes SU FUNCION ES: separar valga la dedundancia los codigos que estean combinados. Esta funcion se debe realizar tanto en el inventario general como en el Inventario de tribu ya que ambos archivos estan incompletos

In [ ]:
# @title
# ============================================
# REFACTOR: extraer códigos y separar columnas
# (entrada por diálogo de subida en Colab)
# Limpia descripciones (sin códigos) y normaliza códigos: "PREF 9.9.9"
# Con opción para procesar VARIOS archivos
# ============================================
!pip -q install unidecode openpyxl

import pandas as pd
import re
import unicodedata

#===== Config =====
HIERARCHY_COLS = ["MEGAPROCESO", "MACROPROCESO", "PROCESO", "SUBPROCESO"]

#Cambia a True si quieres procesar TODOS los archivos subidos en lote
PROCESS_ALL_UPLOADED = False

# ===== Utilidades base =====
def strip_accents(s: str) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)): return ""
    s = str(s)
    return "".join(ch for ch in unicodedata.normalize("NFD", s) if unicodedata.category(ch) != "Mn")

def norm_header(h):
    h = strip_accents(str(h)).upper()
    h = re.sub(r"[^A-Z0-9]+", "_", h)
    h = re.sub(r"_+", "_", h).strip("_")
    return h

def detectar_prefijos_en_df(df_src: pd.DataFrame, columnas):
    patron_prefijo = re.compile(r"^([A-Z]{2,4})\b")
    encontrados = set()
    for col in columnas:
        for val in df_src[col].dropna().astype(str):
            m = patron_prefijo.match(val.strip())
            if m:
                encontrados.add(m.group(1))
    return sorted(encontrados)

def normalize_code_fmt(raw: str) -> str:
    """
    Devuelve código en formato único: 'PREF 1.2.3'
    Acepta variantes con guiones/espacios: 'ES- 2.1', 'ES -2.1.1', 'GOB 1.1', etc.
    """
    if not isinstance(raw, str) or not raw.strip():
        return ""
    s = strip_accents(raw).upper()
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace("-", " ")              # quitar guiones como separadores
    s = re.sub(r"\s+", " ", s).strip()   # colapsar espacios
    m = re.search(r"^([A-Z]{2,4})\s*([0-9]+(?:\.[0-9]+)*)\b", s)
    if not m:
        # fallback: intentar capturar en cualquier posición
        m = re.search(r"([A-Z]{2,4})\s*([0-9]+(?:\.[0-9]+)*)\b", s)
    if m:
        return f"{m.group(1)} {m.group(2)}"
    return s

def process_file(input_path: str) -> str:
    print("\n======================")
    print("Procesando:", input_path)
    df = pd.read_excel(input_path)
    print("Filas originales:", len(df))

    presentes = [c for c in HIERARCHY_COLS if c in df.columns]
    if not presentes:
        raise ValueError("No hay ninguna de las columnas jerárquicas en el archivo.")
    faltantes = [c for c in HIERARCHY_COLS if c not in df.columns]
    if faltantes:
        print(" Columnas no encontradas (se omiten):", faltantes)

    # ---- Prefijos y patrones por archivo ----
    PREFIJOS = detectar_prefijos_en_df(df, presentes)
    print("Prefijos detectados:", PREFIJOS if PREFIJOS else "(ninguno, se usa genérico)")
    prefijos_regex = r"(?:%s)" % "|".join(PREFIJOS) if PREFIJOS else r"(?:[A-Z]{2,4})"
    CODE_PATTERN = re.compile(rf"({prefijos_regex}\s*-?\s*\d+(?:\.\d+)*\b)")
    CODE_ONLY    = re.compile(rf"^\W*({prefijos_regex}\s*-?\s*\d+(?:\.\d+)*\b)")

    def solo_codigo(x: str) -> str:
        """Extrae el código del inicio y lo NORMALIZA a 'PREF 1.2.3'."""
        if not isinstance(x, str):
            return ""
        s = str(x).strip()
        m = CODE_ONLY.match(s)
        return normalize_code_fmt(m.group(1)) if m else ""

    def desc_sin_codigo(x: str) -> str:
        """Devuelve solo la descripción, eliminando el código inicial y separadores (- : —)."""
        if not isinstance(x, str):
            return ""
        s = str(x).strip()
        m = CODE_ONLY.match(s)
        if m:
            rest = s[m.end():]
            rest = re.sub(r"^\s*[-–—:]\s*", "", rest)
            return rest.strip()
        return s

    # ---------- Explosión de SUBPROCESO ----------
    COL_SUB = "SUBPROCESO"
    def explode_subproceso(cell: str):
        """Devuelve lista de (codigo_norm, descripcion). Si no hay códigos, [( '', texto )]."""
        if not isinstance(cell, str) or not cell.strip():
            return []
        s = cell.replace("\r\n", " ").replace("\r", " ").replace("\n", " ")
        s = re.sub(r"\s+", " ", s).strip()
        matches = list(CODE_PATTERN.finditer(s))
        if not matches:
            return [("", s.strip())]
        segs = []
        for i, m in enumerate(matches):
            codigo_raw = m.group(1).strip()
            codigo = normalize_code_fmt(codigo_raw)
            next_start = matches[i+1].start() if i+1 < len(matches) else len(s)
            desc = s[m.end():next_start]
            desc = desc.lstrip(" -–—:").strip()
            segs.append((codigo, desc))
        return segs

    rows = []
    for idx, row in df.iterrows():
        segs = explode_subproceso(row.get(COL_SUB, "")) if COL_SUB in df.columns else []
        if not segs:  # celda vacía o no existe la columna
            base = row.copy()
            base["__ROW_ORIGEN__"] = idx
            base["CODIGO_SUBPROCESO_EXTRAIDO"] = ""
            base["DESCRIPCION_SUBPROCESO_EXTRAIDA"] = ""
            base["REQUIERE_REVISION"] = True
            rows.append(base)
            continue
        any_code = any(c for c, _ in segs)
        for c, d in segs:
            base = row.copy()
            base["__ROW_ORIGEN__"] = idx
            base["CODIGO_SUBPROCESO_EXTRAIDO"] = c                     # <-- ya normalizado
            base["DESCRIPCION_SUBPROCESO_EXTRAIDA"] = d
            base["REQUIERE_REVISION"] = not any_code
            rows.append(base)

    df_exp = pd.DataFrame(rows)

    # ---------- Construir versión "reemplazada" ----------
    # 1) Limpiar descripciones en TODAS las columnas jerárquicas (sin código)
    df_reemplazo = df_exp.copy()
    for col in [c for c in HIERARCHY_COLS if c in df_reemplazo.columns]:
        df_reemplazo[col] = df_reemplazo[col].astype(str).map(desc_sin_codigo)

    # Para SUBPROCESO, usa directamente la descripción extraída (garantiza limpieza)
    if "DESCRIPCION_SUBPROCESO_EXTRAIDA" in df_reemplazo.columns:
        df_reemplazo["SUBPROCESO"] = (
            df_reemplazo["DESCRIPCION_SUBPROCESO_EXTRAIDA"]
            .fillna("").astype(str).str.strip()
        )

    # 2) Validación de posibles “restos” de otro código en la descripción de SUBPROCESO
    if "DESCRIPCION_SUBPROCESO_EXTRAIDA" in df_reemplazo.columns:
        mask_restos = df_reemplazo["DESCRIPCION_SUBPROCESO_EXTRAIDA"].astype(str).str.contains(CODE_PATTERN)
        df_restos = df_reemplazo[mask_restos].copy()
    else:
        df_restos = pd.DataFrame()

    stats = {
        "filas_originales": len(df),
        "filas_resultantes": len(df_reemplazo),
        "filas_para_revision": int(df_reemplazo.get("REQUIERE_REVISION", pd.Series(dtype=bool)).sum()) if "REQUIERE_REVISION" in df_reemplazo else 0,
        "posibles_restos_en_descripcion": len(df_restos)
    }
    print("Stats:", stats)

    # ---------- Insertar 'CODIGO <col>' a la izquierda con formato normalizado ----------
    def insert_codigo_left(df_base: pd.DataFrame, colname: str) -> pd.DataFrame:
        if colname not in df_base.columns:
            return df_base
        code_col = f"CODIGO {colname}"
        serie_codigo = df_exp[colname].astype(str).apply(solo_codigo)  # usar el texto original para extraer
        if code_col in df_base.columns:
            df_base = df_base.drop(columns=[code_col])
        insert_at = df_base.columns.get_loc(colname)
        df_base.insert(insert_at, code_col, serie_codigo)
        return df_base

    for col in presentes:
        df_reemplazo = insert_codigo_left(df_reemplazo, col)

    # Además, aseguramos que CODIGO_SUBPROCESO_EXTRAIDO esté normalizado
    if "CODIGO_SUBPROCESO_EXTRAIDO" in df_reemplazo.columns:
        df_reemplazo["CODIGO_SUBPROCESO_EXTRAIDO"] = df_reemplazo["CODIGO_SUBPROCESO_EXTRAIDO"].map(normalize_code_fmt)

    # ---------- Exportar ----------
    output_path = re.sub(r"\.xlsx$", "_refactorizado.xlsx", input_path, flags=re.I)
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df_reemplazo.to_excel(writer, index=False, sheet_name="SUBPROCESO_reemplazado")
        df_exp.to_excel(writer, index=False, sheet_name="con_columnas_aux")
        df_reemplazo.loc[df_reemplazo.get("REQUIERE_REVISION", False) == True] \
                    .to_excel(writer, index=False, sheet_name="para_revision")

    print(" Archivo generado:", output_path)
    return output_path


# ===== Diálogo de subida (uno o varios archivos) =====
from google.colab import files
print(" Sube uno o varios archivos .xlsx…")
_uploaded = files.upload()
assert _uploaded, "No subiste ningún archivo."
uploaded_files = list(_uploaded.keys())

# Por defecto: solo el PRIMERO. Para lote: pon PROCESS_ALL_UPLOADED=True
files_to_process = uploaded_files if PROCESS_ALL_UPLOADED else [uploaded_files[0]]

out_paths = []
for path in files_to_process:
    out_paths.append(process_file(path))

# Descargar resultados
for out in out_paths:
    files.download(out)

📤 Sube uno o varios archivos .xlsx…


Saving INVENTARIO PROCESOS_REVISIONES CON TRIBUS_Corte AGO06_2025.xlsx to INVENTARIO PROCESOS_REVISIONES CON TRIBUS_Corte AGO06_2025.xlsx

Procesando: INVENTARIO PROCESOS_REVISIONES CON TRIBUS_Corte AGO06_2025.xlsx
Filas originales: 343
Prefijos detectados: ['APO', 'ES', 'GOB', 'OMN', 'OPE', 'SOP']


/tmp/ipython-input-3024533891.py:158: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_restos = df_reemplazo["DESCRIPCION_SUBPROCESO_EXTRAIDA"].astype(str).str.contains(CODE_PATTERN)


Stats: {'filas_originales': 343, 'filas_resultantes': 617, 'filas_para_revision': 0, 'posibles_restos_en_descripcion': 0}
📦 Archivo generado: INVENTARIO PROCESOS_REVISIONES CON TRIBUS_Corte AGO06_2025_refactorizado.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

2da. Parte---Se carga Archivo refactorizado para su consolidazion---

Los archivos son cargados una vez son seleccionados fuera del entorno de desarrollo deben de cargar el archivo a consolidar con el refactor del inventario general + inventario de tribu + inventario de Squad todas las hojas en un mismo archivo en total son 3.

In [7]:
# @title
 # =========================================================
# Inventario + Responsable TRIBU (todas sus columnas) + SQUAD
# Control 1:1 por código y detección flexible de columnas
# =========================================================
!pip -q install rapidfuzz unidecode openpyxl

import re, pandas as pd, numpy as np
from rapidfuzz import fuzz
try:
    from unidecode import unidecode
except ModuleNotFoundError:
    import unicodedata
    def unidecode(s: str) -> str:
        if not isinstance(s, str): return ""
        return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")

# ---------- CONFIG ----------
UMBRAL_MATCH = 92          # >= confiable
MAX_CANDIDATOS = 3
FORMATO_NOMBRES = "title"  # "title" o "upper"

# ---------- Upload ----------
try:
    from google.colab import files
    _EN_COLAB = True
except Exception:
    _EN_COLAB = False

if _EN_COLAB:
    print("👉 Sube tu Excel con 3 hojas (Inventario, Responsable TRIBU, SQUAD)")
    up = files.upload()
    assert up, "No subiste archivo."
    FILE_PATH = list(up.keys())[0]
else:
    FILE_PATH = "/content/INVENTARIO PROCESOS_21ago2025_Tribu_Squad.xlsx"

# ---------- Helpers ----------
def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy(); out.columns = df.columns.str.strip(); return out

def colkey_like(cols, *need):
    for c in cols:
        u = unidecode(c).upper()
        if all(t in u for t in need):
            return c
    return None

def find_code_col(df: pd.DataFrame):
    c = colkey_like(df.columns, "COD", "SUBPROCESO")
    if c is None: c = colkey_like(df.columns, "CODIGO", "SUB")
    return c

def tidy_name_series(s: pd.Series, modo="title") -> pd.Series:
    if modo == "upper":
        return s.fillna("").astype(str).str.strip().str.upper().replace({"": pd.NA})
    _sm = {"de","del","la","las","los","y","e","da","do","dos","das","van","von","san","santa"}
    def _title_es(x):
        w = re.split(r"\s+", str(x).strip().lower()); out=[]
        for i, tok in enumerate(w):
            if not tok: continue
            parts = tok.split("-"); nparts=[]
            for j,p in enumerate(parts):
                if (i>0 or j>0) and p in _sm: nparts.append(p)
                else: nparts.append(p[:1].upper()+p[1:])
            out.append("-".join(nparts))
        return " ".join(out).strip()
    return s.fillna("").astype(str).map(_title_es).replace({"": pd.NA})

ALIAS_EQUIV = {
    "pepe":"jose","paco":"francisco","pancho":"francisco","nacho":"ignacio","chuy":"jesus",
    "chema":"jose maria","toño":"antonio","toni":"antonio","beto":"roberto","lalo":"eduardo",
    "lucho":"luis","juanfer":"juan fernando","juanfe":"juan fernando",
    "ma":"maria","ma.":"maria","mª":"maria","pame":"pamela","pam":"pamela"
}
TITULOS = {"sr","sra","srta","ing","ing.","lic","lic.","dra","dr","dr.","mgs","msc","mg.","abg","arq","eco","eco.","mae","mba","phd","mtr"}

def normalizar_aliases(txt: str) -> str:
    if not isinstance(txt, str) or not txt.strip(): return ""
    s = unidecode(txt).lower().strip()
    toks = [t for t in re.split(r"[^a-z]+", s) if t]
    toks = [t for t in toks if t not in TITULOS]
    toks = [ALIAS_EQUIV.get(t, t) for t in toks]
    STOP = {"de","del","la","las","los","y","e","da","do","dos","das","van","von","san","santa"}
    toks = [t for t in toks if t not in STOP]
    return " ".join(toks)

def preparar_fuzzy_series_alias(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).map(normalizar_aliases)

def apellido_principal(txt: str) -> str:
    s = normalizar_aliases(txt); return s.split()[-1] if s else ""

def expandir_ma_maria_display(s: pd.Series) -> pd.Series:
    pat = re.compile(r'(^|\s)(ma\.|mª)(\s+)', flags=re.IGNORECASE)
    return (s.fillna("").astype(str)
              .map(lambda x: pat.sub(r"\1María\3", x).strip() if x else x)
              .replace({"": pd.NA}))

# ---------- Leer y detectar hojas ----------
sheets_raw = pd.read_excel(FILE_PATH, sheet_name=None)
sheets = {name: normalize_cols(df) for name, df in sheets_raw.items()}

main_name = tribu_name = squad_name = None
for name, df in sheets.items():
    if main_name is None and find_code_col(df): main_name = name
    if tribu_name is None and colkey_like(df.columns, "RESPONSABLE", "TRIBU"): tribu_name = name
    if squad_name is None and (colkey_like(df.columns, "COLABORADOR") or colkey_like(df.columns, "SQUAD") or colkey_like(df.columns, "EQUIPO")):
        squad_name = name

for name in sheets.keys():  # fallbacks por título
    U = unidecode(name).upper()
    if main_name is None and ("INVENTARIO" in U or "PROCESO" in U): main_name = name
    if tribu_name is None and "TRIBU" in U: tribu_name = name
    if squad_name is None and "SQUAD" in U: squad_name = name

assert main_name and tribu_name and squad_name, "No pude identificar todas las hojas (Inventario/Tribu/Squad)."

inv, tribu, squad = sheets[main_name].copy(), sheets[tribu_name].copy(), sheets[squad_name].copy()

# ---------- Columnas clave detectadas ----------
KEY_INV = find_code_col(inv); KEY_TRB = find_code_col(tribu)
RESP_TRIBU_GUESS = colkey_like(tribu.columns, "RESPONSABLE","TRIBU") or colkey_like(tribu.columns, "RESPONSABLE")

COLAB_COL = colkey_like(squad.columns, "COLABORADOR") or "COLABORADOR"
EMAIL_COL = colkey_like(squad.columns, "EMAIL") or "EMAIL"
CARGO_COL = colkey_like(squad.columns, "CARGO") or "CARGO"
SQUAD_COL = (colkey_like(squad.columns, "SQUAD")
             or colkey_like(squad.columns, "EQUIPO", "TRIBU")
             or colkey_like(squad.columns, "EQUIPO")
             or "SQUAD")
DEPTO_COL = colkey_like(squad.columns, "DEPARTAMENTO") or "DEPARTAMENTO"
LINE_MANAGER_COL = colkey_like(squad.columns, "LINE", "MANAGER") or "LINE MANAGER"
VP_COL = colkey_like(squad.columns, "VP") or "VP"

assert KEY_INV in inv.columns and KEY_TRB in tribu.columns, "Faltan columnas de código."

# ---------- 1) Inventario + TRIBU por CÓDIGO (traer TODAS las columnas de TRIBU) ----------
inv["_COD"] = inv[KEY_INV].astype("string").str.strip()
inv = inv.dropna(subset=["_COD"]).drop_duplicates(subset=["_COD"], keep="first")

tribu["_COD"] = tribu[KEY_TRB].astype("string").str.strip()
tribu = tribu.dropna(subset=["_COD"])

# Dejar UNA fila por código y conservar TODAS sus columnas
tribu_full = tribu.sort_values("_COD").drop_duplicates(subset=["_COD"], keep="first").copy()

# Renombrar columnas de TRIBU con sufijo _TRIBU (excepto la que detectamos como responsable)
ren_trb = {}
for c in tribu_full.columns:
    if c == "_COD":
        continue
    u = unidecode(c).upper()
    if RESP_TRIBU_GUESS and u == unidecode(RESP_TRIBU_GUESS).upper():
        ren_trb[c] = "RESPONSABLE_TRIBU"
    else:
        new = f"{c}_TRIBU"
        # evita colisión con columnas del inventario
        while new in inv.columns:
            new += "_TRB"
        ren_trb[c] = new

tribu_take = tribu_full[["_COD"] + [c for c in tribu_full.columns if c != "_COD"]].rename(columns=ren_trb)

# Merge (inventario conserva todas sus columnas + todas las de TRIBU)
inv_tribu = inv.merge(tribu_take, on="_COD", how="left")
# Si no hubiera columna de responsable en TRIBU, créala vacía para los siguientes pasos
if "RESPONSABLE_TRIBU" not in inv_tribu.columns:
    inv_tribu["RESPONSABLE_TRIBU"] = pd.NA

# ---------- 2) TRIBU ↔ SQUAD por NOMBRE (fuzzy) ----------
uniq_resp = (inv_tribu[["RESPONSABLE_TRIBU"]]
             .dropna().drop_duplicates()
             .rename(columns={"RESPONSABLE_TRIBU": "RESP_TRIBU"}))
uniq_resp["_NORM"] = preparar_fuzzy_series_alias(uniq_resp["RESP_TRIBU"])
uniq_resp["_AP"]   = uniq_resp["RESP_TRIBU"].map(apellido_principal)

# Asegura columnas en SQUAD
for need in [COLAB_COL, EMAIL_COL, CARGO_COL, SQUAD_COL, DEPTO_COL, LINE_MANAGER_COL, VP_COL]:
    if need not in squad.columns: squad[need] = pd.NA

squad = squad.copy()
squad["_NORM"] = preparar_fuzzy_series_alias(squad[COLAB_COL])
squad["_AP"]   = squad[COLAB_COL].map(apellido_principal)
squad["_EMAIL_LOCAL"] = squad[EMAIL_COL].fillna("").astype(str).str.split("@").str[0].str.lower()
squad = squad.sort_values([EMAIL_COL], na_position="last").drop_duplicates("_NORM", keep="first")

cand_norm, cand_ap, cand_email = squad["_NORM"].tolist(), squad["_AP"].tolist(), squad["_EMAIL_LOCAL"].tolist()

best_idx, best_score, ambig = [], [], []
for q_norm, q_ap in zip(uniq_resp["_NORM"], uniq_resp["_AP"]):
    if not q_norm or not cand_norm:
        best_idx.append(None); best_score.append(0); ambig.append(False); continue
    scores=[]
    for j, c_norm in enumerate(cand_norm):
        sc = fuzz.token_set_ratio(q_norm, c_norm)
        if q_ap and q_ap == cand_ap[j]: sc += 10
        if cand_email[j] and q_ap and q_ap in cand_email[j]: sc += 3
        scores.append((max(0, min(100, int(sc))), j))
    scores.sort(reverse=True)
    sc_best, j_best = scores[0]
    amb = sum(1 for sc,_ in scores[:MAX_CANDIDATOS] if sc >= UMBRAL_MATCH and (sc_best - sc) <= 2) > 1
    best_idx.append(j_best); best_score.append(sc_best); ambig.append(amb)

match_tbl = uniq_resp.copy()
def takecol(col):
    return [squad[col].iloc[j] if j is not None else pd.NA for j in best_idx]
for col in [COLAB_COL, EMAIL_COL, CARGO_COL, SQUAD_COL, DEPTO_COL, LINE_MANAGER_COL, VP_COL]:
    match_tbl[col+"_SQUAD"] = takecol(col)
match_tbl["SCORE"] = best_score
match_tbl["AMBIGUO"] = ambig

# Presentación nombres
match_tbl["RESP_TRIBU"] = expandir_ma_maria_display(tidy_name_series(match_tbl["RESP_TRIBU"], FORMATO_NOMBRES))
match_tbl[COLAB_COL+"_SQUAD"] = expandir_ma_maria_display(tidy_name_series(match_tbl[COLAB_COL+"_SQUAD"], FORMATO_NOMBRES))
match_tbl = match_tbl.drop_duplicates(subset=["RESP_TRIBU"], keep="first")

# ---------- 3) Volcar matches del SQUAD al INVENTARIO (manteniendo TODO lo de TRIBU) ----------
inv_out = inv_tribu.merge(
    match_tbl.rename(columns={"RESP_TRIBU":"RESPONSABLE_TRIBU"})[
        ["RESPONSABLE_TRIBU","SCORE","AMBIGUO"] +
        [c+"_SQUAD" for c in [COLAB_COL, EMAIL_COL, CARGO_COL, SQUAD_COL, DEPTO_COL, LINE_MANAGER_COL, VP_COL]]
    ],
    on="RESPONSABLE_TRIBU", how="left"
)

rename_map = {
    COLAB_COL+"_SQUAD": "COLABORADOR_SQUAD",
    EMAIL_COL+"_SQUAD": "EMAIL_SQUAD",
    CARGO_COL+"_SQUAD": "CARGO_SQUAD",
    SQUAD_COL+"_SQUAD": "SQUAD_SQUAD",              # o cámbialo a "EQUIPO_SQUAD" si prefieres
    DEPTO_COL+"_SQUAD": "DEPARTAMENTO_SQUAD",
    LINE_MANAGER_COL+"_SQUAD": "LINE_MANAGER_SQUAD",
    VP_COL+"_SQUAD": "VP_SQUAD",
}
inv_out.rename(columns=rename_map, inplace=True)

# Responsable final (prioriza del SQUAD si el match es confiable)
cond_conf = (inv_out["SCORE"].fillna(0) >= UMBRAL_MATCH) & inv_out["COLABORADOR_SQUAD"].notna()
inv_out["RESPONSABLE"] = np.where(cond_conf, inv_out["COLABORADOR_SQUAD"], inv_out["RESPONSABLE_TRIBU"])
inv_out["RESPONSABLE"] = expandir_ma_maria_display(tidy_name_series(inv_out["RESPONSABLE"], FORMATO_NOMBRES))

# ---- Control estricto: exactamente los códigos del inventario ----
cods_inv = set(inv["_COD"].unique())
inv_out = inv_out[inv_out["_COD"].isin(cods_inv)].copy()
inv_out = inv_out.drop_duplicates(subset=["_COD"], keep="first")
esperados = inv["_COD"].nunique()
reales = inv_out["_COD"].nunique()
print(f"Esperados (Inventario): {esperados} | En salida: {reales}")
assert reales == esperados, f"La salida tiene {reales} códigos y el inventario {esperados}"

# ---------- 4) Reporte de revisión ----------
revisar = match_tbl.loc[(match_tbl["SCORE"] < UMBRAL_MATCH) | (match_tbl["AMBIGUO"])].copy()

# ---------- 5) Exportar ----------
OUT_PATH = FILE_PATH.replace(".xlsx", "_MATCHES.xlsx")
with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as w:
    inv_out.to_excel(w, index=False, sheet_name="INVENTARIO_CON_MATCHES")
    match_tbl.to_excel(w, index=False, sheet_name="TRIBU_SQUAD_MATCHES")
    revisar.to_excel(w, index=False, sheet_name="REVISAR_MATCHES")

print("✅ Archivo generado:", OUT_PATH)
if _EN_COLAB:
    files.download(OUT_PATH)

👉 Sube tu Excel con 3 hojas (Inventario, Responsable TRIBU, SQUAD)


Saving INVENTARIO PROCESOS_21ago2025_Tribu_Squad.xlsx to INVENTARIO PROCESOS_21ago2025_Tribu_Squad (3).xlsx
Esperados (Inventario): 605 | En salida: 605
✅ Archivo generado: INVENTARIO PROCESOS_21ago2025_Tribu_Squad (3)_MATCHES.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

IMPORTANTE: UNA VEZ GENERADO EL ARCHIVO CONSOLIDADO CON TRIBU Y SQUAD DEBEN PREPARAR LOS ARCHVIVOS SIGUIENTES DE PRIORIDAD + MATRIZ DE RIESGO
DEJANDO COPIADOS EN UNA HOJA LIMPIA LOS PROCESOS Y SUBPROCESOS DE AMBOS ARCHIVOS PRIORIDAD Y MATRIZ A CONSOLIDAR ALADO DE LA HOJA DE INVENTARIO CONSOLIDADO GENERADO ANTERIORMENTE

3era PARTE --- UNION DE CONSOLIDADO CON TRIBU/SQUAD + PRIORIDAD Y MATRIZ DE RIESGO CON INCONSISTENCIAS SEÑALADAS

In [8]:
# @title
# -*- coding: utf-8 -*-
"""
Consolidador v2 (Refactor): Inventario + Priorización + Matriz de Riesgo
- Mejoras de portabilidad, performance y auditoría
- Unificación de semantics *_MATCH
- Normalización de códigos también en Riesgo
- Vectorización de exact match (merges) y cache de fuzzy
- Hoja adicional con PRIORIZACIONES NO MATCHEADAS (ruta completa)

Requisitos: pandas, numpy, rapidfuzz, openpyxl o xlsxwriter
Ejecuta en Google Colab o localmente (ver sección IO).
"""

import pandas as pd
import numpy as np
import re
import unicodedata
import subprocess
import sys

# ------------------------ Configuración ------------------------
FUZZY_UMBRAL_RIESGO = 88   # 0–100 para SUBPROCESO (Matriz de Riesgo)
FUZZY_UMBRAL_PRIOR  = 88   # 0–100 para PROCEDIMIENTO/SUBPROCESO en Priorización
PRIO_KEYS = ['MEGAPROCESO','MACROPROCESO','PROCESO','COD_PROCEDIMIENTO','PROCEDIMIENTO']  # orden de fuerza
try:
    ENABLE_FUZZY_PRIOR
except NameError:
    ENABLE_FUZZY_PRIOR = True
try:
    ENABLE_FUZZY_RISK
except NameError:
    ENABLE_FUZZY_RISK = True

# Si corres fuera de Colab, define aquí la ruta del archivo entrada
WB_PATH = "input.xlsx"  # cámbialo si no usas Colab

# ------------------------ Utilidades base ------------------------
def strip_accents(s: str) -> str:
    if s is None:
        return None
    s = str(s)
    return ''.join(ch for ch in unicodedata.normalize('NFD', s)
                   if unicodedata.category(ch) != 'Mn')


def _std_na(s):
    """Estandariza nulos comunes representados como strings."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return np.nan
    ss = str(s).strip()
    if ss == '' or ss.lower() in {'nan', 'none', 'null', 'na'}:
        return np.nan
    return ss


def norm_text(x):
    x = _std_na(x)
    if pd.isna(x):
        return np.nan
    x = strip_accents(str(x)).upper().strip()
    x = re.sub(r'\s+', ' ', x)
    return x


def norm_header(h):
    h = strip_accents(str(h)).upper()
    h = re.sub(r'[^A-Z0-9]+', '_', h)
    h = re.sub(r'_+', '_', h).strip('_')
    return h


def find_col(df, candidates):
    norm_map = {norm_header(c): c for c in df.columns}
    for cand in candidates:
        k = norm_header(cand)
        if k in norm_map:
            return norm_map[k]
    return None


def normalize_df_text(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = df[c].map(_std_na).map(norm_text)
    return df


def rename_to_standard(df, mapping_pairs):
    """mapping_pairs = [(col_detected, 'STD_NAME'), ...]"""
    ren = {src: std for (src, std) in mapping_pairs if src}
    return df.rename(columns=ren).copy()


def combined_key(df, keys):
    if not all(k in df.columns for k in keys):
        return None
    parts = [df[k].fillna('') for k in keys]
    return parts[0].astype(str).str.cat([p.astype(str) for p in parts[1:]], sep='||')


def norm_code(x: str) -> str:
    """Normaliza códigos: sin acentos, mayúsculas, sin espacios/puntos/guiones (solo A-Z/0-9)."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ''
    x = strip_accents(str(x)).upper()
    return re.sub(r'[^A-Z0-9]', '', x)


# ------------------------ RapidFuzz (fuzzy) ------------------------
try:
    from rapidfuzz import fuzz, process
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rapidfuzz'])
    from rapidfuzz import fuzz, process


def fuzzy_extract_one(query, choices, scorer=fuzz.token_set_ratio):
    """Retorna (match_text, score_int) o ('', 0) si no hay."""
    if not choices or query is None:
        return ('', 0)
    q = str(query).strip()
    if q == '' or q.lower() in {'nan', 'none', 'null', 'na'}:
        return ('', 0)
    m = process.extractOne(q, choices, scorer=scorer)
    if not m:
        return ('', 0)
    return (m[0], int(m[1]))

# ------------------------ Utilidades de instalación/Excel (Colab-safe) ------------------------
import importlib

def ensure_pkg(pkg: str) -> bool:
    """Intenta importar; si no existe, instala vía pip y reintenta. Devuelve True si queda disponible."""
    try:
        importlib.import_module(pkg)
        return True
    except ModuleNotFoundError:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
            importlib.import_module(pkg)
            return True
        except Exception:
            return False


def write_excel_multi_sheets(out_name: str, sheets: dict) -> str:
    """Escribe un Excel con múltiples hojas usando openpyxl o xlsxwriter, auto-instalando si falta.
    Retorna el engine utilizado o lanza RuntimeError si ambos fallan."""
    # Intentar openpyxl primero (más común para lectura/escritura)
    for engine, pkg in [('openpyxl', 'openpyxl'), ('xlsxwriter', 'xlsxwriter')]:
        if ensure_pkg(pkg):
            try:
                with pd.ExcelWriter(out_name, engine=engine) as writer:
                    for sheet_name, df in sheets.items():
                        df.to_excel(writer, sheet_name=sheet_name, index=False)
                return engine
            except Exception:
                # si falla este engine, probamos el siguiente
                continue
    raise RuntimeError('No se pudo escribir el Excel: faltan engines openpyxl/xlsxwriter o ambos fallaron.')


# ------------------------ Priorización (exacto por código/clave + fuzzy) ------------------------
def _select_best_pri_rows(df: pd.DataFrame, key_cols: list) -> pd.DataFrame:
    """Para registros duplicados por clave, selecciona la fila con mayor 'completitud'.
    Es robusto aunque el DF no tenga ninguna de las columnas en PRIO_KEYS.
    """
    tmp = df.copy()
    # Asegurar columnas de referencia de completitud
    for col in PRIO_KEYS:
        if col not in tmp.columns:
            tmp[col] = np.nan
    present = [c for c in PRIO_KEYS if c in tmp.columns]
    if present:
        tmp['_COMPL_'] = tmp[present].notna().sum(axis=1).astype(int)
    else:
        tmp['_COMPL_'] = 0
    tmp = tmp.sort_values(['_COMPL_'], ascending=[False])
    tmp = tmp.drop_duplicates(subset=key_cols, keep='first')
    tmp = tmp.drop(columns=['_COMPL_'])
    return tmp


def apply_prioritization(inv_orig, pri_orig):
    """
    Devuelve DataFrame con columnas:
      PRIORIZACION, PRIORIZACION_MATCH, PRIORIZACION_SCORE, PRIORIZACION_MATCH_TYPE, PRIORIZACION_QUERY
    Reglas (en orden): exacto por código -> exacto por claves -> fuzzy por nombre.
    Si el catálogo de Priorización no tiene 'PROCEDIMIENTO', usa 'SUBPROCESO'.
    """
    # Detectar columnas equivalentes
    c_inv = {
        'CODIGO_SUBPROCESO': find_col(inv_orig, ['CODIGO SUBPROCESO', 'CODIGOSUBPROCESO', 'COD SUBPROCESO', 'CODIGO_SUBPROCESO']),
        'PROCEDIMIENTO': find_col(inv_orig, ['PROCEDIMIENTO']),
        'SUBPROCESO': find_col(inv_orig, ['SUBPROCESO', 'SUB PROCESO', 'SubProceso']),
        'MEGAPROCESO': find_col(inv_orig, ['MEGAPROCESO', 'MEGAPROCESOS']),
        'MACROPROCESO': find_col(inv_orig, ['MACROPROCESO', 'MACROPROCESOS']),
        'PROCESO': find_col(inv_orig, ['PROCESO', 'PROCESOS'])
    }
    c_pri = {
        'COD_PROCEDIMIENTO': find_col(pri_orig, ['COD PROCEDIMIENTO', 'CODIGO PROCEDIMIENTO', 'COD_PROCEDIMIENTO', 'ID PROCEDIMIENTO']),
        'PROCEDIMIENTO': find_col(pri_orig, ['PROCEDIMIENTO']),
        'SUBPROCESO': find_col(pri_orig, ['SUBPROCESO', 'SUB PROCESO', 'SubProceso']),
        'MEGAPROCESO': find_col(pri_orig, ['MEGAPROCESO', 'MEGAPROCESOS']),
        'MACROPROCESO': find_col(pri_orig, ['MACROPROCESO', 'MACROPROCESOS']),
        'PROCESO': find_col(pri_orig, ['PROCESO', 'PROCESOS']),
        'PRIORIZACION': find_col(pri_orig, ['PRIORIZACION', 'PRIORIZACIÓN'])
    }

    inv_std = rename_to_standard(inv_orig, [(c_inv[k], k) for k in c_inv if c_inv[k]])
    pri_std = rename_to_standard(pri_orig, [(c_pri[k], k) for k in c_pri if c_pri[k]])

    # nombre a usar desde el catálogo de Priorización (puede ser PROCEDIMIENTO o SUBPROCESO)
    pri_name_col = 'PROCEDIMIENTO' if 'PROCEDIMIENTO' in pri_std.columns else ('SUBPROCESO' if 'SUBPROCESO' in pri_std.columns else None)

    # normalizar textos
    inv_std = normalize_df_text(inv_std, ['PROCEDIMIENTO', 'SUBPROCESO', 'MEGAPROCESO', 'MACROPROCESO', 'PROCESO'])
    pri_text_cols = ['MEGAPROCESO', 'MACROPROCESO', 'PROCESO', 'PRIORIZACION'] + ([pri_name_col] if pri_name_col else [])
    pri_std = normalize_df_text(pri_std, [c for c in pri_text_cols if c])

    # normalizar códigos
    inv_std['__CODE__'] = inv_std['CODIGO_SUBPROCESO'].map(norm_code) if 'CODIGO_SUBPROCESO' in inv_std.columns else ''
    pri_std['__CODE__'] = pri_std['COD_PROCEDIMIENTO'].map(norm_code) if 'COD_PROCEDIMIENTO' in pri_std.columns else ''

    out = pd.DataFrame(index=inv_std.index)
    out['PRIORIZACION'] = ''
    out['PRIORIZACION_MATCH'] = ''
    out['PRIORIZACION_SCORE'] = 0
    out['PRIORIZACION_MATCH_TYPE'] = ''

    name_col_inv = 'PROCEDIMIENTO' if 'PROCEDIMIENTO' in inv_std.columns else ('SUBPROCESO' if 'SUBPROCESO' in inv_std.columns else None)
    out['PRIORIZACION_QUERY'] = inv_std[name_col_inv] if name_col_inv else ''

    # 1) EXACTO por CÓDIGO
    if ('__CODE__' in inv_std.columns) and ('__CODE__' in pri_std.columns) and ('PRIORIZACION' in pri_std.columns):
        cols = ['__CODE__', 'PRIORIZACION'] + ([pri_name_col] if pri_name_col else [])
        pr_map_src = pri_std[cols].copy()
        pr_map_src = pr_map_src[~pr_map_src['__CODE__'].isna() & (pr_map_src['__CODE__'] != '')]
        pr_map_src = _select_best_pri_rows(pr_map_src, key_cols=['__CODE__'])
        pr_map = pr_map_src.drop_duplicates('__CODE__')
        m = inv_std[['__CODE__']].merge(pr_map, on='__CODE__', how='left')
        mask = (m['PRIORIZACION'].fillna('') != '')
        out.loc[mask, 'PRIORIZACION'] = m.loc[mask, 'PRIORIZACION'].fillna('')
        if pri_name_col:
            out.loc[mask, 'PRIORIZACION_MATCH'] = m.loc[mask, pri_name_col].fillna('')
        out.loc[mask, 'PRIORIZACION_SCORE'] = 100
        out.loc[mask, 'PRIORIZACION_MATCH_TYPE'] = 'EXACT_CODE'

    # 2) EXACTO por CLAVES
    remaining = out['PRIORIZACION'].astype(str) == ''
    candidate_keys = ['MEGAPROCESO', 'MACROPROCESO', 'PROCESO', 'PROCEDIMIENTO']
    join_keys = [k for k in candidate_keys if (k in inv_std.columns and k in pri_std.columns)]
    if remaining.any() and join_keys and 'PRIORIZACION' in pri_std.columns:
        pri_cols = join_keys + ['PRIORIZACION'] + ([pri_name_col] if pri_name_col else [])
        pri_keys_src = pri_std[pri_cols].copy()
        pri_keys_src = pri_keys_src.dropna(subset=join_keys, how='all')
        pri_keys_src = _select_best_pri_rows(pri_keys_src, key_cols=join_keys)
        j = inv_std.loc[:, join_keys].merge(pri_keys_src, on=join_keys, how='left')
        mask2 = remaining & (j['PRIORIZACION'].fillna('') != '')
        out.loc[mask2, 'PRIORIZACION'] = j.loc[mask2, 'PRIORIZACION'].fillna('')
        if pri_name_col:
            out.loc[mask2, 'PRIORIZACION_MATCH'] = j.loc[mask2, pri_name_col].fillna('')
        out.loc[mask2, 'PRIORIZACION_SCORE'] = 100
        out.loc[mask2, 'PRIORIZACION_MATCH_TYPE'] = 'EXACT_KEY'

    # 3) FUZZY por nombre
    remaining = out['PRIORIZACION'].astype(str) == ''
    if remaining.any() and name_col_inv and ('PRIORIZACION' in pri_std.columns) and pri_name_col and ENABLE_FUZZY_PRIOR:
        pri_pairs = pri_std[[pri_name_col, 'PRIORIZACION']].dropna(subset=[pri_name_col]).drop_duplicates()
        choices = pri_pairs[pri_name_col].tolist()
        prio_by_proc = dict(zip(pri_pairs[pri_name_col], pri_pairs['PRIORIZACION'].fillna('')))

        names = inv_std.loc[remaining, name_col_inv].dropna().drop_duplicates()
        cache = {}
        for nm in names:
            mtxt, sc = fuzzy_extract_one(nm, choices, scorer=fuzz.token_set_ratio)
            cache[nm] = (mtxt, sc) if (sc >= FUZZY_UMBRAL_PRIOR and mtxt in prio_by_proc) else ('', 0)

        q_series = inv_std.loc[remaining, name_col_inv]
        mtxt_series = q_series.map(lambda x: cache.get(x, ('', 0))[0])
        sc_series = q_series.map(lambda x: cache.get(x, ('', 0))[1])
        passed = sc_series >= FUZZY_UMBRAL_PRIOR

        out.loc[remaining & passed, 'PRIORIZACION_MATCH'] = mtxt_series[passed]
        out.loc[remaining & passed, 'PRIORIZACION'] = mtxt_series[passed].map(prio_by_proc)
        out.loc[remaining & passed, 'PRIORIZACION_SCORE'] = sc_series[passed]
        out.loc[remaining & passed, 'PRIORIZACION_MATCH_TYPE'] = 'FUZZY'

    return out


# ------------------------ Matriz de Riesgo (exacto + fuzzy) ------------------------
def apply_risk_matrix(inv_orig, risk_orig):
    """Devuelve DataFrame con:
       MATRIZ DE RIESGOS (SI/NO), SUBPROCESO_MATCH, RISK_SCORE, RISK_MATCH_TYPE
    """
    c_inv_code = find_col(inv_orig, ['CODIGO SUBPROCESO', 'CODIGOSUBPROCESO', 'COD SUBPROCESO', 'CODIGO_SUBPROCESO'])
    c_inv_sub = find_col(inv_orig, ['SUBPROCESO', 'SUB PROCESO', 'SubProceso'])

    c_rk_code = find_col(risk_orig, ['CodigoSubproceso', 'CODIGO SUBPROCESO', 'CODIGOSUBPROCESO', 'COD SUBPROCESO', 'CODIGO_SUBPROCESO'])
    c_rk_sub = find_col(risk_orig, ['Subproceso', 'SUBPROCESO', 'SUB PROCESO', 'SubProceso'])

    inv_std = rename_to_standard(inv_orig, [(c_inv_code, 'CODIGO_SUBPROCESO'), (c_inv_sub, 'SUBPROCESO')])
    rk_std  = rename_to_standard(risk_orig, [(c_rk_code, 'CODIGO_SUBPROCESO'), (c_rk_sub, 'SUBPROCESO')])

    inv_std = normalize_df_text(inv_std, ['CODIGO_SUBPROCESO', 'SUBPROCESO'])
    rk_std  = normalize_df_text(rk_std,  ['CODIGO_SUBPROCESO', 'SUBPROCESO'])

    # Normalizar código de forma estricta
    inv_std['__CODE__'] = inv_std['CODIGO_SUBPROCESO'].map(norm_code) if 'CODIGO_SUBPROCESO' in inv_std.columns else ''
    rk_std['__CODE__']  = rk_std['CODIGO_SUBPROCESO'].map(norm_code) if 'CODIGO_SUBPROCESO' in rk_std.columns else ''

    out = pd.DataFrame(index=inv_std.index)
    out['MATRIZ DE RIESGOS'] = 'NO'
    out['SUBPROCESO_MATCH']  = ''  # texto matched del catálogo de Riesgo
    out['RISK_SCORE']        = 0
    out['RISK_MATCH_TYPE']   = ''   # EXACT, FUZZY, ''

    have_code = ('__CODE__' in inv_std.columns) and ('__CODE__' in rk_std.columns)
    have_sub  = ('SUBPROCESO' in inv_std.columns) and ('SUBPROCESO' in rk_std.columns)

    # ===== Exacto por (codigo + subproceso) =====
    if have_code and have_sub:
        rk_pairs = rk_std[['__CODE__', 'SUBPROCESO']].dropna(how='any').drop_duplicates()
        m = inv_std[['__CODE__', 'SUBPROCESO']].merge(rk_pairs.assign(_HIT_=1), on=['__CODE__', 'SUBPROCESO'], how='left')
        exact_mask = m['_HIT_'].fillna(0).eq(1)
        out.loc[exact_mask, 'MATRIZ DE RIESGOS'] = 'SI'
        out.loc[exact_mask, 'SUBPROCESO_MATCH']  = inv_std.loc[exact_mask, 'SUBPROCESO'].fillna('')
        out.loc[exact_mask, 'RISK_SCORE']        = 100
        out.loc[exact_mask, 'RISK_MATCH_TYPE']   = 'EXACT'

    # ===== Fuzzy por SUBPROCESO, acotando por código cuando se tenga =====
    enable_fuzzy_risk = bool(globals().get('ENABLE_FUZZY_RISK', True))
    if ENABLE_FUZZY_RISK and have_sub:
        remaining = out['MATRIZ DE RIESGOS'] != 'SI'
        if remaining.any():
            # Índice de choices por código si es posible
            choices_by_code = {}
            if have_code:
                tmp = rk_std[['__CODE__', 'SUBPROCESO']].dropna(subset=['SUBPROCESO'])
                for code_val, subdf in tmp.groupby('__CODE__', dropna=False):
                    choices_by_code[code_val] = subdf['SUBPROCESO'].drop_duplicates().tolist()

            # Global fallback
            choices_global = rk_std['SUBPROCESO'].dropna().drop_duplicates().tolist()

            rem_df = inv_std.loc[remaining, ['__CODE__','SUBPROCESO'] if have_code else ['SUBPROCESO']]
            # Cache por (code, subproceso)
            cache = {}
            # Agrupar por código para reducir comparaciones
            if have_code:
                groups = rem_df.groupby('__CODE__', dropna=False)
            else:
                groups = [(None, rem_df)]
            for code_val, g in groups:
                choices = choices_by_code.get(code_val, choices_global)
                names_unique = g['SUBPROCESO'].dropna().drop_duplicates()
                for s in names_unique:
                    mtxt, sc = fuzzy_extract_one(s, choices, scorer=fuzz.token_set_ratio)
                    cache[(code_val, s)] = (mtxt, sc) if sc >= FUZZY_UMBRAL_RIESGO else ('', 0)

            # Mapear
            if have_code:
                mtext = rem_df.apply(lambda r: cache.get((r['__CODE__'], r['SUBPROCESO']), ('', 0))[0], axis=1)
                score = rem_df.apply(lambda r: cache.get((r['__CODE__'], r['SUBPROCESO']), ('', 0))[1], axis=1)
            else:
                mtext = rem_df['SUBPROCESO'].map(lambda x: cache.get((None, x), ('', 0))[0])
                score = rem_df['SUBPROCESO'].map(lambda x: cache.get((None, x), ('', 0))[1])

            passed  = score >= FUZZY_UMBRAL_RIESGO
            idx_rem = rem_df.index
            out.loc[idx_rem[passed], 'MATRIZ DE RIESGOS'] = 'SI'
            out.loc[idx_rem[passed], 'SUBPROCESO_MATCH']  = mtext[passed].values
            out.loc[idx_rem[passed], 'RISK_SCORE']        = score[passed].values
            out.loc[idx_rem[passed], 'RISK_MATCH_TYPE']   = 'FUZZY'

    return out


def risk_exact_diffs(inv_orig, risk_orig):
    """Construye conjuntos EXACTOS (sin fuzzy) para comparar inventario vs catálogo de riesgo.
    Retorna dict con:
      inv_pairs_df: DF con __CODE__, SUBPROCESO y clave exacta __PAIR__ para inventario
      rk_pairs_df : DF con __CODE__, SUBPROCESO y clave exacta __PAIR__ para catálogo
      inv_only_df : Filas del inventario (ruta) cuyo par exacto NO está en catálogo
      rk_only_df  : Filas del catálogo de riesgo cuyo par exacto NO está en inventario
    """
    # Normalizaciones mínimas, espejo de apply_risk_matrix
    c_inv_code = find_col(inv_orig, ['CODIGO SUBPROCESO', 'CODIGOSUBPROCESO', 'COD SUBPROCESO', 'CODIGO_SUBPROCESO'])
    c_inv_sub  = find_col(inv_orig, ['SUBPROCESO', 'SUB PROCESO', 'SubProceso'])
    c_rk_code  = find_col(risk_orig, ['CodigoSubproceso', 'CODIGO SUBPROCESO', 'CODIGOSUBPROCESO', 'COD SUBPROCESO', 'CODIGO_SUBPROCESO'])
    c_rk_sub   = find_col(risk_orig, ['Subproceso', 'SUBPROCESO', 'SUB PROCESO', 'SubProceso'])

    inv_std = rename_to_standard(inv_orig, [(c_inv_code, 'CODIGO_SUBPROCESO'), (c_inv_sub, 'SUBPROCESO')])
    rk_std  = rename_to_standard(risk_orig, [(c_rk_code, 'CODIGO_SUBPROCESO'), (c_rk_sub, 'SUBPROCESO')])
    inv_std = normalize_df_text(inv_std, ['CODIGO_SUBPROCESO', 'SUBPROCESO', 'MEGAPROCESO', 'MACROPROCESO', 'PROCESO', 'PROCEDIMIENTO'])
    rk_std  = normalize_df_text(rk_std,  ['CODIGO_SUBPROCESO', 'SUBPROCESO', 'MEGAPROCESO', 'MACROPROCESO', 'PROCESO'])

    inv_std['__CODE__'] = inv_std['CODIGO_SUBPROCESO'].map(norm_code) if 'CODIGO_SUBPROCESO' in inv_std.columns else ''
    rk_std['__CODE__']  = rk_std['CODIGO_SUBPROCESO'].map(norm_code)  if 'CODIGO_SUBPROCESO' in rk_std.columns else ''

    # pares exactos válidos (requiere código y subproceso no vacíos)
    inv_pairs = inv_std[['__CODE__','SUBPROCESO']].dropna().copy()
    inv_pairs = inv_pairs[(inv_pairs['__CODE__']!='') & (inv_pairs['SUBPROCESO']!='')].drop_duplicates()
    inv_pairs['__PAIR__'] = inv_pairs['__CODE__'].astype(str) + '||' + inv_pairs['SUBPROCESO'].astype(str)

    rk_pairs = rk_std[['__CODE__','SUBPROCESO']].dropna().copy()
    rk_pairs = rk_pairs[(rk_pairs['__CODE__']!='') & (rk_pairs['SUBPROCESO']!='')].drop_duplicates()
    rk_pairs['__PAIR__'] = rk_pairs['__CODE__'].astype(str) + '||' + rk_pairs['SUBPROCESO'].astype(str)

    # Diferencias exactas
    inv_only_keys = set(inv_pairs['__PAIR__']) - set(rk_pairs['__PAIR__'])
    rk_only_keys  = set(rk_pairs['__PAIR__'])  - set(inv_pairs['__PAIR__'])

    # Rutas para reporte
    ruta_cols_inv = [c for c in ['MEGAPROCESO','MACROPROCESO','PROCESO','PROCEDIMIENTO','SUBPROCESO','CODIGO_SUBPROCESO'] if c in inv_std.columns]
    ruta_cols_rk  = [c for c in ['MEGAPROCESO','MACROPROCESO','PROCESO','SUBPROCESO','CODIGO_SUBPROCESO'] if c in rk_std.columns]

    inv_only_df = inv_std.copy()
    inv_only_df['__CODE__'] = inv_only_df['CODIGO_SUBPROCESO'].map(norm_code) if 'CODIGO_SUBPROCESO' in inv_only_df.columns else ''
    inv_only_df['__PAIR__'] = inv_only_df['__CODE__'].astype(str) + '||' + inv_only_df['SUBPROCESO'].astype(str)
    inv_only_df = inv_only_df[inv_only_df['__PAIR__'].isin(inv_only_keys)]
    inv_only_df = inv_only_df[ruta_cols_inv].drop_duplicates()

    rk_only_df = rk_std.copy()
    rk_only_df['__CODE__'] = rk_only_df['CODIGO_SUBPROCESO'].map(norm_code) if 'CODIGO_SUBPROCESO' in rk_only_df.columns else ''
    rk_only_df['__PAIR__'] = rk_only_df['__CODE__'].astype(str) + '||' + rk_only_df['SUBPROCESO'].astype(str)
    rk_only_df = rk_only_df[rk_only_df['__PAIR__'].isin(rk_only_keys)]
    rk_only_df = rk_only_df[ruta_cols_rk].drop_duplicates()

    return {
        'inv_pairs_df': inv_pairs,
        'rk_pairs_df' : rk_pairs,
        'inv_only_df' : inv_only_df,
        'rk_only_df'  : rk_only_df,
    }


# ------------------------ IO: Cargar archivo y hojas ------------------------
def _load_excel_workbook():
    """Carga interactiva en Colab; fuera de Colab usa WB_PATH definido arriba."""
    try:
        from google.colab import files  # type: ignore
        print(" ** Sube el archivo Excel con las 3 hojas (Inventario, Priorización, Matriz de Riesgo)…")
        uploaded = files.upload()
        wb_path = list(uploaded.keys())[0]
        return wb_path, True
    except Exception:
        print(f"Ejecutando fuera de Colab. Usando WB_PATH='{WB_PATH}'. Cambia la ruta si es necesario.")
        return WB_PATH, False


def detect_role_by_name(name):
    n = norm_header(name)
    if 'RIESG' in n:
        return 'risk'
    if 'PRIOR' in n:
        return 'pri'
    # Evitar confundir 'unmatched' como inventario
    if 'INVENT' in n or 'TRIBU' in n or 'SQUAD' in n:
        return 'invent'
    return None


def main():
    wb_path, in_colab = _load_excel_workbook()

    xls = pd.ExcelFile(wb_path)
    sheets = xls.sheet_names
    print("Hojas detectadas:", sheets)

    roles, dfs = {}, {}
    for s in sheets:
        role = detect_role_by_name(s)
        if role and role not in roles:
            roles[role] = s
            dfs[role] = pd.read_excel(wb_path, sheet_name=s, dtype=str)

    # fallback interactivo si faltan hojas
    for role, label in [('invent', 'Inventario'), ('pri', 'Priorización'), ('risk', 'Matriz de Riesgo')]:
        if role not in roles:
            if in_colab:
                chosen = input(f"Escribe el nombre exacto de la hoja para {label}: ").strip()
                roles[role] = chosen
                dfs[role] = pd.read_excel(wb_path, sheet_name=chosen, dtype=str)
            else:
                raise RuntimeError(f"No se detectó la hoja de {label}. Hojas disponibles: {sheets}")

    print("Asignación de hojas:", roles)

    inv_orig = dfs['invent'].copy()
    pri_orig = dfs['pri'].copy()
    risk_orig = dfs['risk'].copy()

    # ------------------------ Procesamiento ------------------------
    prio_out = apply_prioritization(inv_orig, pri_orig)
    risk_out = apply_risk_matrix(inv_orig, risk_orig)

    # Construir salida final sin cambiar orden/filas
    inv_out = inv_orig.copy()
    new_cols = [
        'PRIORIZACION', 'PRIORIZACION_MATCH', 'PRIORIZACION_SCORE', 'PRIORIZACION_MATCH_TYPE', 'PRIORIZACION_QUERY',
        'MATRIZ DE RIESGOS', 'SUBPROCESO_MATCH', 'RISK_SCORE', 'RISK_MATCH_TYPE'
    ]
    for col in new_cols:
        if col not in inv_out.columns:
            inv_out[col] = ''

    inv_out.loc[:, 'PRIORIZACION'] = prio_out['PRIORIZACION'].values
    inv_out.loc[:, 'PRIORIZACION_MATCH'] = prio_out['PRIORIZACION_MATCH'].values
    inv_out.loc[:, 'PRIORIZACION_SCORE'] = prio_out['PRIORIZACION_SCORE'].values
    inv_out.loc[:, 'PRIORIZACION_MATCH_TYPE'] = prio_out['PRIORIZACION_MATCH_TYPE'].values
    inv_out.loc[:, 'PRIORIZACION_QUERY'] = prio_out['PRIORIZACION_QUERY'].values

    inv_out.loc[:, 'MATRIZ DE RIESGOS'] = risk_out['MATRIZ DE RIESGOS'].values
    inv_out.loc[:, 'SUBPROCESO_MATCH'] = risk_out['SUBPROCESO_MATCH'].values
    inv_out.loc[:, 'RISK_SCORE'] = risk_out['RISK_SCORE'].values
    inv_out.loc[:, 'RISK_MATCH_TYPE'] = risk_out['RISK_MATCH_TYPE'].values

    # Orden columnas: originales + nuevas al final
    final_cols = list(inv_orig.columns) + [c for c in new_cols if c in inv_out.columns]
    inv_out = inv_out[final_cols]

    # ------------------------ Hoja de PRIORIZACIONES NO MATCHEADAS ------------------------
    unmatched_mask = inv_out['PRIORIZACION'].astype(str).fillna('').eq('')
    ruta_cols = [c for c in ['MEGAPROCESO', 'MACROPROCESO', 'PROCESO', 'PROCEDIMIENTO', 'SUBPROCESO', 'CODIGO_SUBPROCESO', 'PRIORIZACION_QUERY'] if c in inv_out.columns]
    unmatched_prio = inv_out.loc[unmatched_mask, ruta_cols].copy()
    if 'PRIORIZACION_QUERY' in unmatched_prio.columns:
        unmatched_prio.rename(columns={'PRIORIZACION_QUERY': 'QUERY_UTILIZADA'}, inplace=True)
    unmatched_prio['OBS'] = 'No match en Priorización'

    # ------------------------ Resumen simple ------------------------
    resumen_rows = [
        ['Filas totales', len(inv_out)],
        ['PRIORIZACION asignada', int((~unmatched_mask).sum())],
        ['PRIORIZACION sin asignar', int(unmatched_mask.sum())],
        ["MATRIZ DE RIESGOS = 'SI'", int((inv_out['MATRIZ DE RIESGOS'] == 'SI').sum())]
    ]
    resumen_df = pd.DataFrame(resumen_rows, columns=['Métrica', 'Valor'])

    # ------------------------ Exportar a Excel (múltiples hojas) ------------------------
    out_name = 'INVENTARIO_PRIORIZACION_RIESGO_v2_Colab.xlsx'

    # Diferenciales EXACTOS de Riesgo (sin fuzzy):
    diffs = risk_exact_diffs(inv_orig, risk_orig)
    inv_only_exact = diffs['inv_only_df'].copy()
    rk_only_exact  = diffs['rk_only_df'].copy()
    inv_only_exact['OBS'] = 'Inventario sin Matriz (EXACT)'
    rk_only_exact['OBS']  = 'Matriz de Riesgo sin Inventario (EXACT)'

    # Resumen extendido
    resumen_rows = [
        ['Filas totales Consolidado', len(inv_out)],
        ['PRIORIZACION asignada', int((~unmatched_mask).sum())],
        ['PRIORIZACION sin asignar', int(unmatched_mask.sum())],
        ["MATRIZ DE RIESGOS = 'SI' (con fuzzy)", int((inv_out['MATRIZ DE RIESGOS'] == 'SI').sum())],
        ['Riesgo EXACT match (pares código+subproceso)', int(len(set(diffs['inv_pairs_df']['__PAIR__']) & set(diffs['rk_pairs_df']['__PAIR__'])))],
        ['Inventario sin Matriz (EXACT)', len(inv_only_exact)],
        ['Matriz de Riesgo sin Inventario (EXACT)', len(rk_only_exact)],
    ]
    resumen_df = pd.DataFrame(resumen_rows, columns=['Métrica', 'Valor'])

    engine_used = write_excel_multi_sheets(out_name, {
        'Consolidado': inv_out,
        'Unmatched_Priorizacion': unmatched_prio,
        'Inv_Sin_Matriz_EXACT': inv_only_exact,
        'Matriz_Sin_Inv_EXACT': rk_only_exact,
        'Resumen': resumen_df,
    })

    print(f"Archivo generado: {out_name} (engine: {engine_used})")

    # Descarga en Colab
    if in_colab:
        try:
            from google.colab import files  # type: ignore
            files.download(out_name)
        except Exception:
            pass


if __name__ == '__main__':
    main()

 ** Sube el archivo Excel con las 3 hojas (Inventario, Priorización, Matriz de Riesgo)…


Saving INVENTARIO PROCESOS_21ago2025_Tribu_Squad_MATCHES_PRIORIZACION-MATRIZ_DE_RIESGO.xlsx to INVENTARIO PROCESOS_21ago2025_Tribu_Squad_MATCHES_PRIORIZACION-MATRIZ_DE_RIESGO.xlsx
Hojas detectadas: ['INVENTARIO_CON_MATCHES TRIBU-SQ', 'PRIORIZACION', 'MATRIZ DE RIESGO']
Asignación de hojas: {'invent': 'INVENTARIO_CON_MATCHES TRIBU-SQ', 'pri': 'PRIORIZACION', 'risk': 'MATRIZ DE RIESGO'}
Archivo generado: INVENTARIO_PRIORIZACION_RIESGO_v2_Colab.xlsx (engine: openpyxl)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

4ta. PARTE --- VALIDACION DE DIFERENCIAS ENTRE MAPA EN RDS E INVENTARIO ACTUALIZADO

En esta validacion -Prepara las columnas de ambos archivos tanto el del mapa de procesos que se descarga en RDS como el inventario actualizado || Desde MACROPROCESOS  hasta SUBPROCESOS || añadiento las columnas || NOMBRES DE TRIBUS, NOMBRES DE SQUAD Y MATRIZ DE RIESGO || Para la comparacion de cambios a realizar

In [ ]:
# ============================================================
# RECONCILIACIÓN MAPA vs INVENTARIO (Colab) - Versión Final
# - 0_Resumen
# - 1_Eliminar_MAPA  (incluye Observacion + DifCampos)
# - 2_Añadir_MAPA
# - 3_Cambiar_MAPA   (incluye DifCampos + LocalizacionCambio + Observacion)
# - 4_Exactos
# ============================================================

from google.colab import files
import pandas as pd
import numpy as np
from unicodedata import normalize
import io
from datetime import datetime

# ---------- Subir archivo ----------
print("📂 Selecciona tu archivo Excel (hojas: INVENTARIO DE PROCEDIMIENTOS, MAPA DE PROCEDIMIENTOS)")
uploaded = files.upload()
xlsx_name = list(uploaded.keys())[0]
xlsx_stream = io.BytesIO(uploaded[xlsx_name])

# ---------- Funciones ----------
def clean_text(s):
    if pd.isna(s): return np.nan
    s = str(s)
    s = normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = s.strip().lower()
    s = " ".join(s.split())
    return s

def normalize_df_columns(df):
    out = df.copy()
    out.columns = [clean_text(c) for c in out.columns]
    return out

def first_existing(df, options):
    for op in options:
        if op in df.columns:
            return op
    return None

def build_standard_view(df, source):
    dfN = normalize_df_columns(df)
    opts = {
        "codigo_macroproceso": ["codigo macroproceso"],
        "macroproceso":       ["macroproceso", "macro proceso"],
        "codigo_proceso":     ["codigo proceso"],
        "proceso":            ["proceso"],
        "codigo_subproceso":  ["codigo subproceso"],
        "subproceso":         ["subproceso", "sub proceso"],
    }
    std = dfN.copy()
    for std_col, op_list in opts.items():
        found = first_existing(std, op_list)
        std[std_col] = std[found] if found else np.nan

    # clave de reconciliación
    if std["codigo_subproceso"].notna().any():
        std["key_subproceso"] = std["codigo_subproceso"].apply(clean_text)
    else:
        std["key_subproceso"] = (
            std["codigo_proceso"].astype(str).apply(clean_text).fillna("") + " | " +
            std["subproceso"].astype(str).apply(clean_text).fillna("")
        ).str.strip(" | ")

    for col in ["codigo_macroproceso","macroproceso","codigo_proceso","proceso","codigo_subproceso","subproceso"]:
        std[f"norm_{col}"] = std[col].apply(clean_text)

    std["_source"] = source
    return std

# ---------- Leer archivo ----------
xls = pd.ExcelFile(xlsx_stream)
inv_raw  = pd.read_excel(xls, sheet_name="INVENTARIO DE PROCEDIMIENTOS")
mapa_raw = pd.read_excel(xls, sheet_name="MAPA DE PROCEDIMIENTOS")

inv  = build_standard_view(inv_raw,  "INV")
mapa = build_standard_view(mapa_raw, "MAPA")

pairs = [
    ("codigo_macroproceso","CODIGO_MACROPROCESO"),
    ("macroproceso","MACROPROCESO"),
    ("codigo_proceso","CODIGO_PROCESO"),
    ("proceso","PROCESO"),
    ("codigo_subproceso","CODIGO_SUBPROCESO"),
    ("subproceso","SUBPROCESO"),
]

merged = mapa.merge(inv, on="key_subproceso", how="outer", suffixes=("_MAPA","_INV"), indicator=True)

# Comparaciones campo a campo
cmp_cols = []
for base, label in pairs:
    merged[f"cmp_{base}"] = merged[f"norm_{base}_MAPA"] == merged[f"norm_{base}_INV"]
    cmp_cols.append(f"cmp_{base}")
all_equal = merged[cmp_cols].all(axis=1)

# ---------- Exactos ----------
exactos = merged[(merged["_merge"]=="both") & (all_equal)].copy()
exactos_out = exactos[[f"{b}_MAPA" for b,_ in pairs] + [f"{b}_INV" for b,_ in pairs]]

# ---------- Eliminar (con Observacion + DifCampos) ----------
elim = merged[merged["_merge"]=="left_only"].copy()
elim_out = pd.DataFrame()
observaciones = []
difcampos = []

# preconjunto de valores del inventario para match por campo
inv_norm_sets = {base: set(inv[f"norm_{base}"].dropna().unique()) for base, _ in pairs}

for idx, row in elim.iterrows():
    matches = []
    diffs = []
    # Guardar valores lado MAPA + INVmatch (valor si aparece en inventario, blanco si no)
    for base, label in pairs:
        val_m = row.get(f"{base}_MAPA")
        norm_m = clean_text(val_m)
        elim_out.loc[idx, f"{label}_MAPA"] = val_m
        elim_out.loc[idx, f"{label}_INVmatch"] = val_m if (norm_m in inv_norm_sets[base]) else ""
        if norm_m in inv_norm_sets[base]:
            matches.append(label)
        else:
            diffs.append(label)
    observaciones.append(("Coinciden: " + ", ".join(matches)) if matches else "Sin coincidencias")
    difcampos.append(", ".join(diffs))

elim_out["Observacion"] = observaciones
elim_out["DifCampos"] = difcampos

# ---------- Añadir ----------
anadir = merged[merged["_merge"]=="right_only"].copy()
anadir_out = anadir[[f"{b}_INV" for b,_ in pairs]]

# ---------- Cambiar (con DifCampos + LocalizacionCambio + Observacion) ----------
cambiar = merged[(merged["_merge"]=="both") & (~all_equal)].copy()
cambiar_out = cambiar[[f"{b}_MAPA" for b,_ in pairs] + [f"{b}_INV" for b,_ in pairs]].copy()

def diff_fields(row):
    diffs = []
    locs = []
    equals = []
    for base, label in pairs:
        val_m = str(row.get(f"{base}_MAPA")).strip().lower() if pd.notna(row.get(f"{base}_MAPA")) else ""
        val_i = str(row.get(f"{base}_INV")).strip().lower() if pd.notna(row.get(f"{base}_INV")) else ""
        if val_m != "" and val_i != "":
            if val_m != val_i:
                diffs.append(label)
                locs.append(f"{label}_MAPA vs {label}_INV")
            else:
                equals.append(label)
    return ", ".join(diffs), "; ".join(locs), ("Coinciden en: " + ", ".join(equals) if equals else "Sin coincidencias")

cambiar_out["DifCampos"], cambiar_out["LocalizacionCambio"], cambiar_out["Observacion"] = zip(*cambiar.apply(diff_fields, axis=1))

# ---------- Resumen ----------
resumen = pd.DataFrame([{
    "Total_MAPA": int(mapa["key_subproceso"].nunique()),
    "Total_INVENTARIO": int(inv["key_subproceso"].nunique()),
    "Exactos": int(len(exactos_out)),
    "Eliminar_MAPA": int(len(elim_out)),
    "Añadir_MAPA": int(len(anadir_out)),
    "Cambiar_MAPA": int(len(cambiar_out)),
    "Check_INVENTARIO (Exactos + Cambiar + Añadir)": int(len(exactos_out) + len(cambiar_out) + len(anadir_out))
}])

# ---------- Guardar ----------
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_name = f"reporte_final_resumen_{ts}.xlsx"

with pd.ExcelWriter(out_name, engine="openpyxl") as writer:
    resumen.to_excel(writer, index=False, sheet_name="0_Resumen")
    elim_out.to_excel(writer, index=False, sheet_name="1_Eliminar_MAPA")
    anadir_out.to_excel(writer, index=False, sheet_name="2_Añadir_MAPA")
    cambiar_out.to_excel(writer, index=False, sheet_name="3_Cambiar_MAPA")
    exactos_out.to_excel(writer, index=False, sheet_name="4_Exactos")

print("✅ Archivo generado:", out_name)
files.download(out_name)
